In [1]:
import pandas as pd
import unicodedata

# diacritics as their unicode value
LOTONE = chr(0x0300)
HITONE = chr(0x0301)
RISETONE = chr(0x030C)
MIDTONE1 = chr(0x0304)
MIDTONE2 = chr(0x0305)
TONECHARS = {LOTONE, HITONE, RISETONE,MIDTONE1,MIDTONE2}

UNDERDOT = chr(0x0323)
UNDERLINE = chr(0x0329)
UNDERDIACS = {UNDERDOT, UNDERLINE}

# split characters into letters (with their diacritics)
def get_letters(text):
    try:
        text = unicodedata.normalize('NFD', text)
    except:
        print(text)
        return []
    text = text.replace(UNDERLINE, UNDERDOT)
    letters = []
    i = 0
    while i < len(text):
        curr_letter = ''
        # look for gb
        if ((i+1) < len(text) and ((text[i] == 'g') and (text[i+1] == 'b'))):
            curr_letter = text[i:i+2]
            i+=2

        # check if next char exists and is a diacritic
        elif ((i+1) < len(text)) and ((text[i+1] in TONECHARS) or text[i+1] in UNDERDIACS):
            if ((i+2) < len(text) and ((text[i+2] in TONECHARS) or text[i+2] in UNDERDIACS)):
                curr_letter = text[i:i+3]
                # print(f"{text[i:i+3]}\t{text[max(i-4, 0):i+4]}\t{text}")
                i+=3 # skip next two chars
            else:
                curr_letter = text[i:i+2]
                i+=2 # skip next char
                
        # normal case (the letter is one single char)
        else: 
            curr_letter = text[i]
            i+=1 # go to next char
        
        # add letter to list
        letters.append(curr_letter.lower())
    return letters

# load in data
def load_dataset(filename):
    return pd.read_csv(filename, header=0, index_col=0)

In [29]:
iroyin_full = load_dataset('/Users/graven2/Documents/THESIS/data/iroyinspeech_full2_deduped.csv')
multidiac_full = load_dataset('/Users/graven2/Documents/THESIS/data/multidiac_full.csv')
yad_full = load_dataset('/Users/graven2/Documents/THESIS/data/yad_full_CLEAN.csv')

In [6]:
display(iroyin_full)

,sentence
id,
00001.wav,Nàìjíríà yóò gba ikọ̀ ọmọ ogun tuntun.
00002.wav,Wọ́n fẹ́ gbógun ti ìdúnkòkò àti ètò ààbò tó ń ...
00003.wav,Ọ̀jọ̀gbọ́n Yẹmí Òṣínbàjò jẹ́ igbákejì ààrẹ.
00004.wav,Mo lọ sí ìlú Àbújá lọ́jọ́ Ajé láti pàdé ọ...
00005.wav,Ọ̀jọ̀gbọ́n Ajáńlékokò sọ̀rọ̀ lásìkò ìpàdé ...
...,...
yo_p_4496.wav,"Ohun pàtàkì ni kí obìnrin mọ iná dá, ṣu..."
yo_p_4497.wav,Ìròrí àwọn ọmọ tálákà ni pé àwọn ọmọ ...
yo_p_4498.wav,Adéọlá sùn kalẹ̀ nítorí kò fẹ́ tẹ̀lé ì...


In [30]:
all_data = pd.DataFrame(columns=['Source', 'ID', 'Sentence'])
print('---IROYIN---')
for index, row in iroyin_full.iterrows():
    all_data.loc[len(all_data)] = ['IroyinSpeech', index, row['sentence']]

print('---MultiDiac---')
for index, row in multidiac_full.iterrows():
    all_data.loc[len(all_data)] = ['MultiDiac', index, row['sentence']]

print('---YAD---')
for index, row in yad_full.iterrows():
    all_data.loc[len(all_data)] = ['YAD', index, row['sentence']]


---IROYIN---
---MultiDiac---
---YAD---


In [31]:
def dup_rows(df):
    print(f'Original Length: {len(df)}')
    dup_sentences = df.duplicated(subset='Sentence', keep=False)
    dup_indices = dup_sentences[dup_sentences == True]
    print(f'{len(dup_indices)} TOTAL DUPLICATES FOUND')
    for i in dup_indices.index:
        print(f'{i}: {df.loc[i]}')
    dropped = df.drop_duplicates(subset='Sentence',keep='first')
    print(f'New Length: {len(dropped)}')
    return dropped

deduped = dup_rows(all_data)

Original Length: 43494
48 TOTAL DUPLICATES FOUND
249: Source                                           IroyinSpeech
ID                                                  00250.wav
Sentence    A ti gbé ìgbésẹ̀ láti bá àwọn tí ọ̀rọ̀ náà kàn...
Name: 249, dtype: object
253: Source                                           IroyinSpeech
ID                                                  00254.wav
Sentence    A ó túbọ̀ máa fọwọ́sowọ̀pọ́ pẹ̀lú ìjọba ìpínlẹ...
Name: 253, dtype: object
278: Source                                           IroyinSpeech
ID                                                  00279.wav
Sentence    Ọjọ́ kẹ́rìnlélógún, oṣú kejìlá, ọdún yìí ní il...
Name: 278, dtype: object
312: Source                                           IroyinSpeech
ID                                                  00313.wav
Sentence    Ó tún wá dúpẹ́ lọ́wọ́ ilé ìgbìmọ̀ asòfin fún à...
Name: 312, dtype: object
2133: Source                                   IroyinSpeech
ID                         

In [32]:
display(deduped)

,Source,ID,Sentence
0,IroyinSpeech,00001.wav,Nàìjíríà yóò gba ikọ̀ ọmọ ogun tuntun.
1,IroyinSpeech,00002.wav,Wọ́n fẹ́ gbógun ti ìdúnkòkò àti ètò ààbò tó ń ...
2,IroyinSpeech,00003.wav,Ọ̀jọ̀gbọ́n Yẹmí Òṣínbàjò jẹ́ igbákejì ààrẹ.
3,IroyinSpeech,00004.wav,Mo lọ sí ìlú Àbújá lọ́jọ́ Ajé láti pàdé ọ...
4,IroyinSpeech,00005.wav,Ọ̀jọ̀gbọ́n Ajáńlékokò sọ̀rọ̀ lásìkò ìpàdé ...
...,...,...,...
43489,YAD,20117,"Màdáámú, ẹ wo bí."
43490,YAD,20118,"“Omijé náà ti ṣé ìran Làbákẹ́ bàìbàì, láàrin ì..."
43491,YAD,20119,"Orílẹ̀-èdè sí orílẹ̀-èdè, àjọ ilẹ̀ òkèèrè, Àjọ..."
43492,YAD,20121,Èyíkéyìí agbára tàbí ohun-èlò tí wọ́n bá gbà g...


In [23]:
# get first 90% of each corpus for train
full_ir = deduped[deduped['Source'] == 'IroyinSpeech']
slicer = len(full_ir) - int(len(full_ir)/10)
print(len(full_ir), slicer)
train_ir = full_ir.iloc[:slicer]
test_ir = full_ir.iloc[slicer:]
# get last 10% of each corpus for test

26688 24020


In [33]:
test_set_idx = (
    deduped.groupby('Source', group_keys=False)
    .apply(lambda x: x.tail(int(len(x) * 0.1)))
    .index
)

test_set = deduped.loc[test_set_idx]
train_set = deduped[~deduped.index.isin(test_set_idx)]

# no need for val set, so it can overlap with train
val_set = (
    train_set.groupby('Source', group_keys=False)
    .apply(lambda x: x.tail(int(len(x) * 0.1)))
    .reset_index(drop=True)
)

print(len(test_set), len(train_set), len(val_set))
display(test_set, train_set)

4345 39125 3912


/var/folders/nx/n0bhmnq56qs0j8dw9w1xnwmm0000gp/T/ipykernel_21872/4238042184.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  deduped.groupby('Source', group_keys=False)
/var/folders/nx/n0bhmnq56qs0j8dw9w1xnwmm0000gp/T/ipykernel_21872/4238042184.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_set.groupby('Source', group_keys=False)


,Source,ID,Sentence
24020,IroyinSpeech,yo_p_1084.wav,Amẹ́ríkà yóò ran Nàíjíríà lọ́wọ́ nípa ètò aj...
24021,IroyinSpeech,yo_p_1086.wav,Bàbá náà sọ̀rọ̀ yìí ní ibi ayẹyẹ àádọ́rin...
24022,IroyinSpeech,yo_p_1088.wav,Olùkọ́ fún àwọn akẹ́kọ̀ọ́ ní iṣẹ́-àmúrel...
24023,IroyinSpeech,yo_p_1090.wav,Adarí fún ọ̀rọ̀ ọ òkèèrè ilẹ̀ Nàìjíríà tẹpẹl...
24024,IroyinSpeech,yo_p_1096.wav,A mọ rírì i rẹ̀ torí náà ni a ṣe ń jó ti...
...,...,...,...
43489,YAD,20117,"Màdáámú, ẹ wo bí."
43490,YAD,20118,"“Omijé náà ti ṣé ìran Làbákẹ́ bàìbàì, láàrin ì..."
43491,YAD,20119,"Orílẹ̀-èdè sí orílẹ̀-èdè, àjọ ilẹ̀ òkèèrè, Àjọ..."
43492,YAD,20121,Èyíkéyìí agbára tàbí ohun-èlò tí wọ́n bá gbà g...


,Source,ID,Sentence
0,IroyinSpeech,00001.wav,Nàìjíríà yóò gba ikọ̀ ọmọ ogun tuntun.
1,IroyinSpeech,00002.wav,Wọ́n fẹ́ gbógun ti ìdúnkòkò àti ètò ààbò tó ń ...
2,IroyinSpeech,00003.wav,Ọ̀jọ̀gbọ́n Yẹmí Òṣínbàjò jẹ́ igbákejì ààrẹ.
3,IroyinSpeech,00004.wav,Mo lọ sí ìlú Àbújá lọ́jọ́ Ajé láti pàdé ọ...
4,IroyinSpeech,00005.wav,Ọ̀jọ̀gbọ́n Ajáńlékokò sọ̀rọ̀ lásìkò ìpàdé ...
...,...,...,...
41863,YAD,18018,Nítorí náà tí ẹ bá rí obìnrin kan pẹ̀lú irun k...
41864,YAD,18020,"Àwa rè é, tí à ń pílè ohùn oyin padà fún ẹgbẹ́..."
41865,YAD,18021,Orílẹ̀-èdè méjèèjì jọ ní àdéhùn pé wọn yóò jọ ...
41866,YAD,18022,Ó fikún-un pé àwo̩n méjéèjì fún àwo̩n gómìnà n...


In [28]:
# put into LLM format
train_set['Sentence'].to_csv('train.txt', index=False, header=False)
test_set['Sentence'].to_csv('test.txt', index=False, header=False)
val_set['Sentence'].to_csv('dev.txt', index=False, header=False)
